# Informe de un cliente — 2025 vs 2026

Ejecute todo en orden (*Run All*). El notebook tiene dos partes:

- **A. Preparación**: parámetros, conexión, el Parquet de enero 2025 al último mes cerrado y su lectura.
- **C. Salidas**: el PowerPoint del cliente, desde sus líneas de factura (se consultan en C.4).

Los cuatro notebooks (`Informe_Ajustes_Region`, `Res`, `Top_Clientes` e `Informe_Cliente`) salen de `Descuentos.ipynb` y cada uno trae su propia copia de la preparación y de las funciones que usa: un cambio en una función compartida hay que repetirlo en los notebooks que la tengan. Los Parquet son los mismos para todos: si otro notebook ya los generó con las mismas fechas, ponga `GENERAR_PARQUET = False` (y `GENERAR_PARQUET_D = False`, si aplica) para no volver a consultar la base.

# A. Preparación
## A.1 Parámetros

> **Cliente del informe: `CLIENTE`**.

In [ ]:
from datetime import date, timedelta
from pathlib import Path

AQUI = Path.cwd()
SALIDA = AQUI / "output" / "Informe_Cliente"   # carpeta donde se guardan los resultados de este notebook
SALIDA.mkdir(parents=True, exist_ok=True)

# Base: Parquet de enero 2025 a septiembre 2026 (se genera en A.3)
FECHA_INICIO = date(2025, 1, 1)
FECHA_FIN = date(2026, 10, 16)         # EXCLUSIVO
PARQUET = AQUI / f"Descuentos_{FECHA_INICIO:%Y%m%d}_{FECHA_FIN:%Y%m%d}.parquet"
GENERAR_PARQUET = True               # True: lo vuelve a consultar aunque ya exista (toma varios minutos)

# >>> MES A REVISAR EN CLIENTES <<<   1 = enero ... 12 = diciembre; None = el ultimo mes con datos
MES_CLIENTES = 10

CLIENTE = "1200110538"                # cliente del informe individual (C.4)
TOP_RANGOS = 8                        # grupos de material y distribuidoras con grafica de rangos
TOP_MATERIALES = 10                   # materiales en el informe del cliente

## A.2 Librerías y conexión

Usa la conexión `DB_*` del `.env` de esta carpeta o, si no hay, el del proyecto de lista de precios.

In [ ]:
import base64
import io
import os
from contextlib import contextmanager

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from sqlalchemy import URL, create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def engine():
    load_dotenv(AQUI / ".env") or load_dotenv(AQUI.parent / "Lista de precios" / ".env")
    faltan = [v for v in ("DB_HOST", "DB_NAME", "DB_USER", "DB_PASSWORD") if not os.getenv(v)]
    if faltan:
        raise RuntimeError(f"Faltan en el .env: {', '.join(faltan)}. "
                           f"Copie .env.example como .env en {AQUI} y completelo.")
    url = URL.create("postgresql+psycopg2", username=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
                     host=os.environ["DB_HOST"], port=int(os.getenv("DB_PORT", "5432")),
                     database=os.environ["DB_NAME"])
    return create_engine(url, pool_pre_ping=True)


ENGINE = engine()
with ENGINE.connect() as c:
    c.execute(text("SELECT 1"))
print(f"Conexion OK: {ENGINE.url.host}/{ENGINE.url.database}")

## A.3 Generación del Parquet

Consulta `commercial_bi` (vista `daily_sales_sector_10_15_20_21`) de `FECHA_INICIO` a `FECHA_FIN` y escribe el
Parquet por partes de `FILAS_POR_PARTE` filas (son ~10 millones). **Solo consulta si el archivo no existe o si
`GENERAR_PARQUET = True`**; si no, usa el que hay.

- `value` es la venta **neta**; bruta = `value + discount_value`. `discount_value` es el ajuste total.
- Fecha corta = `short_dated_discount`; ajuste de precio de mercado = `discount_value − short_dated_discount`.
- Grano: fecha × cliente × grupo de material 6 × distribuidora × canal × sector. El % de lista y el rango (22
  tramos) se calculan sobre ese grano.

In [ ]:
FILAS_POR_PARTE = 500_000

SQL_DESCUENTOS = """
WITH cliente_attr AS (
    -- Un unico registro por cliente (misma regla que lista de precios:
    -- price_group valido y el menor alfabeticamente).
    SELECT DISTINCT ON (cc.client)
        cc.client,
        NULLIF(cc.price_group, '')       AS price_group,
        cc.name                          AS nombre_cliente,
        NULLIF(cc.customer_typology, '') AS tipologia
    FROM public.commercial_clients cc
    ORDER BY cc.client,
             (cc.price_group IS NULL OR cc.price_group = '') ASC,
             cc.price_group ASC
)
SELECT
    v.date                                           AS fecha,
    v.client                                         AS cliente,
    cp.nombre_cliente,
    cp.price_group                                   AS lista_precio,
    m.gr6                                            AS grupo_material_6,
    g6.name                                          AS nombre_gr6,
    dc.distribution_channel_name                     AS canal_distribucion,
    cdd.new_zone_code                                AS distribuidora,
    regexp_replace(cdd.new_zone_name, '^[^-]*-', '') AS nombre_distribuidora,
    cdd.company_code::text                           AS sociedad,
    -- La tipologia de la venta es la del momento; si viene vacia, la del maestro
    LPAD(COALESCE(NULLIF(v.client_typology, ''), cp.tipologia, '0'), 3, '0') AS tipologia,
    t.name                                           AS nombre_tipologia,
    t.sales_channel_name                             AS canal_venta,
    m.sector,
    SUM(v.kilogram)                                  AS venta_kg,
    SUM(v.value)                                     AS venta_neta,
    SUM(v.value + COALESCE(v.discount_value, 0))     AS venta_bruta,
    SUM(COALESCE(v.discount_value, 0))               AS descuento_total,
    SUM(v.discount_market_price_adjustment)          AS descuento_ajuste_mercado,
    SUM(COALESCE(v.short_dated_discount, 0))         AS descuento_fecha_corta,
    SUM(COALESCE(v.focus_discount, 0))               AS descuento_foco
FROM public.daily_sales_sector_10_15_20_21 v
LEFT JOIN public.materials m
       ON v.material::integer = m.code
LEFT JOIN public.material_groups_6 g6
       ON g6.code = m.gr6
LEFT JOIN public.commercial_distributors_details cdd
       ON v.id_commercial_distributors = cdd.id
LEFT JOIN public.distribution_channels dc
       ON v.distribution_channel = dc.distribution_channel
LEFT JOIN cliente_attr cp
       ON v.client = cp.client
LEFT JOIN public.vw_dim_tipologias_cliente t
       ON t.client_typology_id = LPAD(COALESCE(NULLIF(v.client_typology, ''), cp.tipologia, '0'), 3, '0')
WHERE v.date >= :fecha_inicio
  AND v.date <  :fecha_fin                 -- fin EXCLUSIVO
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14
HAVING COALESCE(SUM(v.value), 0) <> 0
    OR COALESCE(SUM(v.kilogram), 0) <> 0
ORDER BY fecha, cliente, grupo_material_6;
"""

# Rangos: (limite inferior del % de lista, rotulo), de mayor a menor. El numero del rotulo ordena los tramos.
RANGOS = [(100, "100% O MAS"), (97, "Mas de 97 menos de 100%"), (95, "Mas de 95 menos de 97%"),
          (93, "Mas de 93 menos de 95%"), (90, "Mas de 90 menos de 93%"), (87, "Mas de 87 menos de 90%"),
          (85, "Mas de 85 menos de 87%"), (83, "Mas de 83 menos de 85%"), (80, "Mas de 80 menos de 83%"),
          (77, "Mas de 77 menos de 80%"), (75, "Mas de 75 menos de 77%"), (73, "Mas de 73 menos de 75%"),
          (70, "Mas de 70 menos de 73%"), (67, "Mas de 67 menos de 70%"), (65, "Mas de 65 menos de 67%"),
          (63, "Mas de 63 menos de 65%"), (60, "Mas de 60 menos de 63%"), (57, "Mas de 57 menos de 60%"),
          (55, "Mas de 55 menos de 57%"), (53, "Mas de 53 menos de 55%"), (50, "Mas de 50 menos de 53%")]
_ROTULOS = [f"{i:02d}. {r}" for i, (_, r) in enumerate(RANGOS, start=1)] + [f"{len(RANGOS) + 1:02d}. Menos de 50%"]
_ROTULO_SIN_BRUTA = f"{len(RANGOS) + 2:02d}. Sin venta bruta"
_BORDES = [float("-inf")] + [limite for limite, _ in reversed(RANGOS)] + [float("inf")]
MOTIVOS_FILA = {"descuento_fecha_corta": "Fecha corta", "descuento_ajuste_mercado": "Ajuste mercado",
                "descuento_foco": "Foco"}
COLUMNAS = {
    "fecha": "Fecha", "cliente": "Codigo SAP Cliente", "nombre_cliente": "Nombre Cliente",
    "lista_precio": "Lista de Precios", "venta_kg": "Venta Kg", "venta_neta": "Venta Neta",
    "venta_bruta": "Venta Bruta", "descuento_total": "Valor Total Descuento",
    "descuento_ajuste_mercado": "Descuento Ajuste Precio Mercado",
    "descuento_fecha_corta": "Descuento Rotacion / Fecha Corta", "descuento_foco": "Descuento Foco",
    "pct_descuento_total": "% Descuento Total", "pct_descuento_ajuste": "% Descuento Ajuste Mercado",
    "pct_descuento_fecha_corta": "% Descuento Rotacion / Fecha Corta", "pct_descuento_foco": "% Descuento Foco",
    "precio_promedio": "Precio Promedio", "grupo_material_6": "Grupo Material 6", "nombre_gr6": "Nombre GR6",
    "canal_distribucion": "Canal de Distribucion", "distribuidora": "Distribuidora",
    "nombre_distribuidora": "Nombre Distribuidora", "sociedad": "Sociedad", "tipologia": "Tipologia Cliente",
    "nombre_tipologia": "Nombre Tipologia", "canal_venta": "Canal de Venta", "sector": "Sector",
    "pct_lista": "% Lista Total", "pct_lista_rotacion": "% Lista Dcto Rotacion",
    "pct_lista_ajuste": "% Lista Dcto Ajuste", "pct_lista_foco": "% Lista Dcto Foco",
    "rango": "Rango Total", "rango_rotacion": "Rango Dcto Rotacion", "rango_ajuste": "Rango Dcto Ajuste",
    "rango_foco": "Rango Dcto Foco", "motivo": "Motivo Descuento",
}


def rango(pct_: pd.Series) -> pd.Series:
    """% de lista (0 a 100) -> rotulo del tramo. [97, 100) es 'Mas de 97 menos de 100%'."""
    tramos = pd.cut(pct_, bins=_BORDES, labels=list(reversed(_ROTULOS)), right=False)
    return tramos.astype(str).where(pct_.notna(), _ROTULO_SIN_BRUTA)


def calcular(df: pd.DataFrame) -> pd.DataFrame:
    """Medidas, % sobre la venta bruta, % de lista, rangos y motivo de cada fila."""
    medidas = ["venta_kg", "venta_neta", "venta_bruta", "descuento_total",
               "descuento_ajuste_mercado", "descuento_fecha_corta", "descuento_foco"]
    df[medidas] = df[medidas].astype(float)
    df["precio_promedio"] = (df["venta_neta"] / df["venta_kg"].where(df["venta_kg"] != 0)).round(0)
    bruta = df["venta_bruta"].where(df["venta_bruta"] > 0)   # devoluciones (bruta <= 0): sin %

    def sobre_bruta(serie: pd.Series) -> pd.Series:
        return (serie / bruta * 100).round(2)

    df["pct_descuento_total"] = sobre_bruta(df["descuento_total"])
    df["pct_descuento_ajuste"] = sobre_bruta(df["descuento_ajuste_mercado"])
    df["pct_descuento_fecha_corta"] = sobre_bruta(df["descuento_fecha_corta"])
    df["pct_descuento_foco"] = sobre_bruta(df["descuento_foco"])
    # % de lista: el total usa la neta; los otros aislan un solo descuento, como si fuera el unico
    df["pct_lista"] = sobre_bruta(df["venta_neta"])
    df["pct_lista_rotacion"] = sobre_bruta(df["venta_bruta"] - df["descuento_fecha_corta"])
    df["pct_lista_ajuste"] = sobre_bruta(df["venta_bruta"] - df["descuento_ajuste_mercado"])
    df["pct_lista_foco"] = sobre_bruta(df["venta_bruta"] - df["descuento_foco"])
    for sufijo in ("", "_rotacion", "_ajuste", "_foco"):
        df["rango" + sufijo] = rango(df["pct_lista" + sufijo])
    partes = [df[col].gt(0).map({True: nombre + " + ", False: ""}) for col, nombre in MOTIVOS_FILA.items()]
    texto_ = pd.concat(partes, axis=1).sum(axis=1).str.removesuffix(" + ")
    df["motivo"] = texto_.where(texto_ != "", "Sin descuento")
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df[list(COLUMNAS)].rename(columns=COLUMNAS)


def tipo_parquet(serie: pd.Series) -> pa.DataType:
    if pd.api.types.is_datetime64_any_dtype(serie):
        return pa.timestamp("ms")
    if pd.api.types.is_numeric_dtype(serie):
        return pa.float64()
    return pa.string()


def escribir_parquet(partes, salida: Path) -> int:
    """Escribe parte por parte. El esquema se fija con la primera parte (texto en lo que no es numero ni fecha).
    Si algo falla a mitad, borra el archivo: uno incompleto es peor que ninguno."""
    escritor, filas = None, 0
    try:
        for df in partes:
            if escritor is None:
                esquema = pa.schema([(c, tipo_parquet(df[c])) for c in df.columns])
                escritor = pq.ParquetWriter(salida, esquema)
            escritor.write_table(pa.Table.from_pandas(df, schema=esquema, preserve_index=False))
            filas += len(df)
            print(f"  {filas:,} filas")
    except BaseException:
        if escritor is not None:
            escritor.close()
            salida.unlink(missing_ok=True)
        raise
    if escritor is not None:
        escritor.close()
    return filas


if GENERAR_PARQUET or not PARQUET.exists():
    print(f"Consultando {FECHA_INICIO} -> {FECHA_FIN} (exclusivo)...")
    with ENGINE.connect() as c:
        c = c.execution_options(stream_results=True)   # el servidor entrega las filas por partes
        partes = pd.read_sql(text(SQL_DESCUENTOS), c, chunksize=FILAS_POR_PARTE,
                             params={"fecha_inicio": FECHA_INICIO, "fecha_fin": FECHA_FIN})
        escribir_parquet((calcular(p) for p in partes), PARQUET)
print(f"Base: {PARQUET.name}")

## A.5 Nombres, formato y funciones comunes

In [ ]:
SOCIEDADES = {"1000": "Región Oriente", "2000": "Región Occidente"}   # company_code: 1000 Región Oriente, 2000 Región Occidente
VISTAS = {"Región Oriente": "Región Oriente", "Región Occidente": "Región Occidente"}
ARCHIVO = {"Región Oriente": "Region_Oriente", "Región Occidente": "Region_Occidente"}   # nombre en los archivos
CORTO = {soc: vista for vista, soc in VISTAS.items()}
NOMBRE_CANAL_VENTA = {"Directo": "Amas de casa"}
CANALES_VENTA_TOP = ["Tradicional", "Institucional", "Modernos", "Amas de casa"]   # orden en tablas y top
ROTULO_CANAL = {"Mayoreo": "Mayoreo", "Puntos Venta": "Puntos de Venta"}   # canal de distribucion
# Canales que siempre facturan al 100% de la lista (se senalan en los hallazgos)
ESPECIALES = ("SUMINISTRO", "GERENCIA", "POLLO CANAL")
# Dimensiones de los rangos (secciones 3 a 6 y 8): rotulo -> columna del Parquet (None = el total)
DIMENSIONES = {"Total": None,
               "Canal de distribución": "Canal de Distribucion",
               "Canal de venta": "Canal de Venta",
               "Grupo de material 6": "Nombre GR6",
               "Distribuidora": "Nombre Distribuidora"}
# Cruce con el canal de distribucion (seccion 7): rotulo -> columna del Parquet
CRUCES = {"Canal de venta": "Canal de Venta"}
TOPE = {"Grupo de material 6": TOP_RANGOS, "Distribuidora": TOP_RANGOS}
CON_CALOR = {"Grupo de material 6"}   # tablas resumen con mapa de calor en los % de ajuste
MEDIDAS = ["Venta Kg", "Venta Bruta", "Venta Neta",
           "Descuento Ajuste Precio Mercado", "Descuento Rotacion / Fecha Corta"]

# --- nombres normalizados
VENTA_BRUTA, VENTA_NETA, VENTA_KG = "Venta bruta ($)", "Venta neta ($)", "Venta kg"
PRECIO_LISTA, PRECIO_NETO = "Precio lista por kg", "Precio neto por kg"
L_GEN, D_GEN = "% aplicado al precio de lista general", "% Ajuste general"
L_AJ, D_AJ = "% aplicado al precio de lista ajuste de precio de mercado", "% Ajuste de precio de mercado"
L_FC, D_FC = "% aplicado al precio de lista fecha corta", "% Ajuste de fecha corta"
P_GEN, P_AJ, P_FC = "Ajuste general ($)", "Ajuste de precio de mercado ($)", "Ajuste de fecha corta ($)"
MILLON = 1     # divisor de los montos: 1 = pesos completos (1e6 = millones)
PILAR = [L_GEN, D_GEN, L_AJ, D_AJ, L_FC, D_FC]
BLOQUE = [VENTA_BRUTA, VENTA_NETA, VENTA_KG, PRECIO_LISTA, PRECIO_NETO,
          L_GEN, D_GEN, P_GEN, L_AJ, D_AJ, P_AJ, L_FC, D_FC, P_FC]
COMPACTO = [VENTA_KG, VENTA_BRUTA, P_GEN, VENTA_NETA, PRECIO_NETO, *PILAR]

# --- colores: motivo y tipo de ajuste comparten color (ajuste = verde, fecha corta = amarillo)
MOTIVOS = {"Con fecha corta": "#eda100", "Solo ajuste de precio de mercado": "#1baf7a", "Sin ajuste": "#b5b4af"}
COLOR_LISTA = {L_GEN: "#52514e", L_AJ: "#1baf7a", L_FC: "#eda100"}
CORTO_LISTA = {L_GEN: "General", L_AJ: "Ajuste de precio de mercado", L_FC: "Fecha corta"}
COLOR_SOC = {"Región Oriente": "#2a78d6", "Región Occidente": "#eb6834"}
TINTE_SOC = {"Región Oriente": "#a9c9f0", "Región Occidente": "#f5b49a"}

TRAMOS = [r[4:] for r in _ROTULOS]  # "100% O MAS", ..., "Menos de 50%"
LIMITE = dict(zip(_ROTULOS, [lim for lim, _ in RANGOS] + [float("-inf")]))
MESES_ES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
            "septiembre", "octubre", "noviembre", "diciembre"]


# --- formato colombiano
def num(v, dec: int = 1) -> str:
    """Miles con punto, decimales con coma (3.831,3)."""
    if pd.isna(v):
        return "-"
    return f"{v:,.{dec}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def pct(v) -> str:
    return "-" if pd.isna(v) else num(v) + "%"


def pesos(v) -> str:
    return "-" if pd.isna(v) else "$" + num(v, 0)


def variacion(v, en_puntos: bool) -> str:
    """+5,9% (variacion) o -6,4 pts (diferencia entre porcentajes)."""
    if pd.isna(v):
        return "-"
    return ("+" if v >= 0 else "-") + num(abs(v)) + (" pts" if en_puntos else "%")


def formato_de(indicador: str):
    if indicador.startswith("%"):
        return pct
    if indicador.startswith("Venta kg"):   # kilos completos, sin decimales
        return lambda v: num(v, 0)
    if indicador.startswith("Precio") or indicador.endswith("($)"):
        return pesos
    return num


# --- indicadores (todo desde sumas, nunca promediando % de filas)
def indicadores(s: pd.DataFrame) -> pd.DataFrame:
    """s: sumas de MEDIDAS (cualquier indice) -> BLOQUE de indicadores."""
    bruta = s["Venta Bruta"]
    b = bruta.where(bruta != 0)
    kg = s["Venta Kg"].where(s["Venta Kg"] != 0)
    aj, fc = s["Descuento Ajuste Precio Mercado"], s["Descuento Rotacion / Fecha Corta"]
    gen = aj + fc
    return pd.DataFrame({
        VENTA_BRUTA: bruta / MILLON,
        VENTA_NETA: s["Venta Neta"] / MILLON,
        VENTA_KG: s["Venta Kg"],
        PRECIO_LISTA: bruta / kg,
        PRECIO_NETO: s["Venta Neta"] / kg,
        L_GEN: (bruta - gen) / b * 100,
        D_GEN: gen / b * 100,
        P_GEN: gen / MILLON,
        L_AJ: (bruta - aj) / b * 100,
        D_AJ: aj / b * 100,
        P_AJ: aj / MILLON,
        L_FC: (bruta - fc) / b * 100,
        D_FC: fc / b * 100,
        P_FC: fc / MILLON,
    }, index=s.index)


def comparar(t: pd.DataFrame) -> pd.DataFrame:
    """t: filas = años -> filas = indicadores; columnas 2025, 2026 y Variación (% o pts)."""
    t = t.T
    out = t[[A1, A2]].copy()
    out.columns = [str(A1), str(A2)]
    es_pct = out.index.str.startswith("%")
    out["Variación"] = ((out[str(A2)] / out[str(A1)] - 1) * 100).where(~es_pct, out[str(A2)] - out[str(A1)])
    return out


def es_variacion(col) -> bool:
    """Columna de variación entre años: 'Variación' en cualquier nivel del encabezado."""
    return any(str(x).startswith("Variación") for x in (col if isinstance(col, tuple) else (col,)))


def estilo_filas(t: pd.DataFrame, titulo: str):
    """Tabla con indicadores en las filas: formato por fila; las columnas 'Variación' en % o pts."""
    variaciones = [c for c in t.columns if es_variacion(c)]
    valores = [c for c in t.columns if c not in variaciones]
    est = t.style
    for fila in t.index:
        est = est.format(formato_de(fila), subset=pd.IndexSlice[[fila], valores])
        if variaciones:
            est = est.format(lambda v, p=fila.startswith("%"): variacion(v, p), subset=pd.IndexSlice[[fila], variaciones])
    return est.set_caption(titulo)


def estilo_columnas(t: pd.DataFrame, titulo: str):
    """Tabla con indicadores en las columnas (nivel 0 si son varias): formato por columna."""
    est = t.style
    for c in t.columns:
        if pd.api.types.is_numeric_dtype(t[c]):   # las columnas de texto (p. ej. canal de venta) van tal cual
            est = est.format(formato_de(c[0] if isinstance(c, tuple) else c), subset=[c])
    return est.set_caption(titulo)


def lado_a_lado(fig, estilo):
    """Grafica y tabla una al lado de la otra (se apilan si no caben)."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    img = base64.b64encode(buf.getvalue()).decode()
    display(HTML('<div style="display:flex;gap:18px;align-items:flex-start;flex-wrap:wrap">'
                 f'<img style="max-width:62%;min-width:520px" src="data:image/png;base64,{img}"/>'
                 f'<div style="font-size:11px">{estilo.to_html()}</div></div>'))


def ejes_limpios(ax, formato_y=None, formato_x=None):
    ax.grid(axis="y" if formato_y else "x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(colors="#52514e", labelsize=8)
    if formato_y:
        ax.yaxis.set_major_formatter(formato_y)
    if formato_x:
        ax.xaxis.set_major_formatter(formato_x)

## A.6 Lectura del Parquet

Una sola pasada por lotes (~10 millones de filas) que deja tablas pequeñas:

- `mensual_soc`: sociedad × mes, **todos los meses** (panorama y evolución).
- `cubo`: dimensión × miembro × sociedad × año × rango × motivo, **enero al último mes con datos** (rangos).
- `cruce`: canal de venta por canal de distribución (sección 7).
- `clientes` y `clientes_det`: **solo el mes de `MES_CLIENTES`** y el mismo mes del año anterior (sección 9 y top
  de clientes).

La sociedad sale de `company_code`: si el Parquet no trae la columna `Sociedad`, se asigna por distribuidora.

In [ ]:
archivo = pq.ParquetFile(PARQUET)
# Ultimo dia con datos (de las estadisticas del archivo, sin leerlo)
i_fecha = archivo.schema_arrow.get_field_index("Fecha")
FECHA_MAX = pd.Timestamp(max(archivo.metadata.row_group(i).column(i_fecha).statistics.max
                             for i in range(archivo.metadata.num_row_groups)))
A2 = FECHA_MAX.year
A1 = A2 - 1
ANIOS = [A1, A2]
MES_ULTIMO = FECHA_MAX.month                         # cierre del acumulado
MESES_ACUM = list(range(1, MES_ULTIMO + 1))
PERIODO = f"ene–{MESES_ES[MES_ULTIMO - 1][:3]}"      # "ene–sep"
MES_FIN_TXT = MESES_ES[MES_ULTIMO - 1]               # "septiembre"
MES_CLI = MES_CLIENTES or MES_ULTIMO                 # mes de los clientes (A.1)
MES_TXT = MESES_ES[MES_CLI - 1]

columnas = ["Fecha", "Distribuidora", *MEDIDAS, "Rango Total", "Codigo SAP Cliente", "Nombre Cliente",
            *[c for c in DIMENSIONES.values() if c]]
if "Sociedad" in archivo.schema_arrow.names:
    columnas.append("Sociedad")
    sociedad_de = None
else:
    with ENGINE.connect() as c:
        sociedad_de = pd.read_sql(text(
            "SELECT DISTINCT new_zone_code, company_code::text AS sociedad "
            "FROM public.commercial_distributors_details"), c).set_index("new_zone_code")["sociedad"]
    assert sociedad_de.index.is_unique, "Hay distribuidoras en mas de una sociedad"

partes_mes, partes_cubo, partes_cli, nombres, partes_cv, partes_cruce, partes_det = [], [], [], [], [], [], []
for lote in archivo.iter_batches(batch_size=FILAS_POR_PARTE, columns=columnas):
    d = lote.to_pandas()
    if sociedad_de is not None:
        d["Sociedad"] = d["Distribuidora"].map(sociedad_de)
    d["Sociedad"] = d["Sociedad"].map(SOCIEDADES).fillna(d["Sociedad"]).fillna("Sin sociedad")
    d = d[d["Sociedad"].isin(VISTAS.values())]
    d["Canal de Venta"] = d["Canal de Venta"].replace(NOMBRE_CANAL_VENTA)
    d["Mes"] = d["Fecha"].dt.to_period("M")
    partes_mes.append(d.groupby(["Sociedad", "Mes"])[MEDIDAS].sum())

    d["Año"] = d["Fecha"].dt.year
    d["Motivo"] = (pd.Series("Sin ajuste", index=d.index)
                   .mask(d["Descuento Ajuste Precio Mercado"] > 0, "Solo ajuste de precio de mercado")
                   .mask(d["Descuento Rotacion / Fecha Corta"] > 0, "Con fecha corta"))

    acum = d[d["Fecha"].dt.month.isin(MESES_ACUM) & d["Año"].isin(ANIOS)]
    for dim, col in DIMENSIONES.items():
        miembro = (acum[col].fillna("(vacío)") if col else pd.Series("Total", index=acum.index)).rename("Miembro")
        partes_cubo.append(pd.concat(
            {dim: acum.groupby([miembro, "Sociedad", "Año", "Rango Total", "Motivo"])[MEDIDAS].sum()},
            names=["Dimension"]))

    for dim, col in CRUCES.items():
        partes_cruce.append(pd.concat(
            {dim: acum.groupby([acum[col].fillna("(vacío)").rename("Miembro"), "Canal de Distribucion",
                                "Sociedad", "Año"])[MEDIDAS].sum()}, names=["Dimension"]))

    mes_cli = d[(d["Fecha"].dt.month == MES_CLI) & d["Año"].isin(ANIOS)]
    partes_cli.append(mes_cli.groupby(["Codigo SAP Cliente", "Sociedad", "Canal de Distribucion", "Año"])[MEDIDAS].sum())
    nombres.append(mes_cli.groupby("Codigo SAP Cliente")["Nombre Cliente"].first())
    partes_cv.append(mes_cli[mes_cli["Año"] == A2].groupby(["Codigo SAP Cliente", "Canal de Venta"])["Venta Bruta"].sum())
    partes_det.append(mes_cli[mes_cli["Año"] == A2].fillna({"Canal de Venta": "(sin canal)", "Nombre Distribuidora": "(vacío)"})
                      .groupby(["Codigo SAP Cliente", "Sociedad", "Nombre Distribuidora", "Canal de Distribucion",
                                "Canal de Venta"])[MEDIDAS].sum())


def juntar(partes: list) -> pd.DataFrame:
    t = pd.concat(partes)
    return t.groupby(level=list(range(t.index.nlevels))).sum()


mensual_soc = juntar(partes_mes)          # sociedad x mes
cubo = juntar(partes_cubo)
cruce = juntar(partes_cruce)              # canal de venta x canal de distribucion x sociedad x año
clientes = juntar(partes_cli)             # cliente x sociedad x canal de distribucion x año (mes de clientes)
clientes_det = juntar(partes_det)         # cliente x distribuidora x canales (mes de clientes, ultimo año)
nombre_cliente = pd.concat(nombres)
nombre_cliente = nombre_cliente[~nombre_cliente.index.duplicated()]
# canal de venta de cada cliente en el mes (si tuvo varios, el de mas venta bruta)
canal_venta_cliente = (pd.concat(partes_cv).groupby(level=[0, 1]).sum().reset_index()
                       .sort_values("Venta Bruta").drop_duplicates("Codigo SAP Cliente", keep="last")
                       .set_index("Codigo SAP Cliente")["Canal de Venta"])
del partes_mes, partes_cubo, partes_cli, nombres, partes_cv, partes_cruce, partes_det

print(f"Datos hasta {FECHA_MAX:%d/%m/%Y} · acumulado {PERIODO} {A1} vs {A2} · clientes de {MES_TXT} | "
      f"cubo {len(cubo):,} filas · clientes {len(clientes):,} filas")

# C. Salidas

Requieren las partes A: aquí solo se generan los archivos, en `SALIDA`:

| Sección | Archivo |
|---|---|
| C.4 Un cliente | `Informe_Cliente_<código>_<año>_vs_<año>.pptx` |

## C.1 Funciones comunes de PowerPoint

In [ ]:
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt


def rgb(h: str) -> RGBColor:
    return RGBColor.from_string(h.lstrip("#").upper())


TINTA, TINTA_2, TINTA_3 = rgb("#0b0b0b"), rgb("#52514e"), rgb("#8c8b87")
FONDO_ENC, BLANCO, ACENTO = rgb("#f0efeb"), rgb("#ffffff"), rgb("#2a78d6")
ANCHO, ALTO, MARGEN = Inches(13.333), Inches(7.5), Inches(0.5)
UTIL_W = ANCHO - 2 * MARGEN
CUERPO_Y = Inches(1.6)
CUERPO_H = ALTO - CUERPO_Y - Inches(0.55)
prs = None   # presentacion en construccion


def nueva_presentacion():
    global prs
    prs = Presentation()
    prs.slide_width, prs.slide_height = ANCHO, ALTO


def guardar(nombre: str):
    archivo_ = SALIDA / nombre
    try:
        prs.save(archivo_)
        print(f"{archivo_.name}: {len(prs.slides)} láminas")
    except PermissionError:
        print(f"No se pudo guardar {archivo_.name}: ciérrelo en PowerPoint y vuelva a correr esta celda.")


def caja_texto(d, x, y, w, h, parrafos, tam=14, color=TINTA, negrita=False, alinear=PP_ALIGN.LEFT, espacio=0):
    """parrafos: texto o lista; cada parrafo es texto o lista de (texto, negrita)."""
    caja = d.shapes.add_textbox(x, y, w, h)
    tf = caja.text_frame
    tf.word_wrap = True
    for i, par in enumerate([parrafos] if isinstance(parrafos, str) else parrafos):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = alinear
        p.space_after = Pt(espacio)
        for trozo, neg in ([(par, negrita)] if isinstance(par, str) else par):
            r = p.add_run()
            r.text = trozo
            r.font.size, r.font.bold = Pt(tam), neg
            r.font.color.rgb = color
    return caja


def diapositiva(titulo: str, subtitulo: str = None, seccion: str = None):
    """Lamina con seccion (arriba), titular (el hallazgo) y bajada."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    if seccion:
        caja_texto(d, MARGEN, Inches(0.12), UTIL_W, Inches(0.25), seccion.upper(), tam=9, color=ACENTO, negrita=True)
    caja_texto(d, MARGEN, Inches(0.32), UTIL_W, Inches(0.8), titulo, tam=19, negrita=True)
    if subtitulo:
        caja_texto(d, MARGEN, Inches(1.13), UTIL_W, Inches(0.4), subtitulo, tam=12, color=TINTA_2)
    return d


def portada(titulo: str, linea_1: str, linea_2: str, color: str):
    """Titulo, barra de color, dos lineas de contexto y el area al pie."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    caja_texto(d, MARGEN, Inches(2.2), UTIL_W, Inches(0.9), titulo, tam=28, negrita=True)
    barra = d.shapes.add_shape(1, MARGEN, Inches(3.2), Inches(1.2), Inches(0.06))
    barra.fill.solid()
    barra.fill.fore_color.rgb = rgb(color)
    barra.line.fill.background()
    caja_texto(d, MARGEN, Inches(3.4), UTIL_W, Inches(0.4), linea_1, tam=18, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(4.0), UTIL_W, Inches(0.34), linea_2, tam=14, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(6.5), UTIL_W, Inches(0.4), "Sistemas de Información", tam=12, color=TINTA_3)


def nombre_sociedad(soc: str) -> str:
    """Nombre de la region para las portadas."""
    return soc


def imagen(d, fig, x=MARGEN, y=CUERPO_Y, w=UTIL_W, h=CUERPO_H):
    """Figura ajustada a la caja w x h, sin deformarla y centrada en horizontal."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    pic = d.shapes.add_picture(buf, x, y)
    escala = min(w / pic.width, h / pic.height)
    pic.width, pic.height = int(pic.width * escala), int(pic.height * escala)
    pic.left = int(x + (w - pic.width) / 2)
    return pic


def color_calor(k: float) -> RGBColor:
    """Escala de un solo tono: blanco (k=0) a naranja (k=1), con texto oscuro legible en todo el rango."""
    a, b = (255, 255, 255), (0xE0, 0x6A, 0x2E)
    k = 0.06 + 0.7 * max(0.0, min(1.0, k))
    return RGBColor(*[int(x + (y - x) * k) for x, y in zip(a, b)])


def tabla(d, df: pd.DataFrame, x=MARGEN, y=CUERPO_Y, w=UTIL_W, tam=10, ancho_indice=0.25,
          negrita_filas=(), alto_fila=None, calor=None, anchos=None):
    """Tabla nativa con el texto ya formateado. Columnas de 2 niveles -> 2 filas de encabezado (combinadas).
    calor: DataFrame numerico del mismo tamaño; las columnas con valores se pintan como mapa de calor. La escala es
    por columna, o por indicador (todos sus años juntos) si las columnas tienen 2 niveles.
    anchos: ancho de cada columna en pulgadas, incluida la del indice (si no, se reparten por igual)."""
    if calor is not None:
        calor = calor.astype(float).copy()
        grupos = (calor.columns.get_level_values(0) if isinstance(calor.columns, pd.MultiIndex)
                  else calor.columns)
        for g in pd.unique(grupos):
            cols_g = [col for col, gg in zip(calor.columns, grupos) if gg == g]
            bloque = calor[cols_g]
            lo, hi = bloque.min().min(), bloque.max().max()
            calor[cols_g] = (bloque - lo) / ((hi - lo) or 1)
    dos = isinstance(df.columns, pd.MultiIndex)
    n_enc = 2 if dos else 1
    n_f, n_c = df.shape[0] + n_enc, df.shape[1] + 1
    alto_fila = alto_fila or Inches(0.08 + tam * 0.021)
    forma = d.shapes.add_table(n_f, n_c, x, y, w, alto_fila * n_f)
    t = forma.table
    t.first_row, t.horz_banding = True, False
    t.columns[0].width = int(w * ancho_indice)
    for j in range(1, n_c):
        t.columns[j].width = int(w * (1 - ancho_indice) / (n_c - 1))
    for j, a_ in enumerate(anchos or []):
        t.columns[j].width = Inches(a_)
    for i in range(n_f):
        t.rows[i].height = alto_fila
        for j in range(n_c):
            celda_ = t.cell(i, j)
            if i < n_enc:
                if j == 0:
                    texto_ = (df.index.name or "") if i == n_enc - 1 else ""
                elif dos:
                    texto_ = str(df.columns[j - 1][i])
                else:
                    texto_ = str(df.columns[j - 1])
            else:
                texto_ = str(df.index[i - n_enc]) if j == 0 else str(df.iat[i - n_enc, j - 1])
            celda_.text = texto_
            celda_.fill.solid()
            celda_.fill.fore_color.rgb = FONDO_ENC if i < n_enc else BLANCO
            if calor is not None and i >= n_enc and j > 0 and pd.notna(calor.iat[i - n_enc, j - 1]):
                celda_.fill.fore_color.rgb = color_calor(calor.iat[i - n_enc, j - 1])
            celda_.vertical_anchor = MSO_ANCHOR.MIDDLE
            celda_.margin_left = celda_.margin_right = Inches(0.04)
            celda_.margin_top = celda_.margin_bottom = Inches(0.01)
            p = celda_.text_frame.paragraphs[0]
            p.alignment = PP_ALIGN.LEFT if j == 0 else (PP_ALIGN.CENTER if i < n_enc else PP_ALIGN.RIGHT)
            for r in p.runs:
                r.font.size = Pt(tam)
                r.font.bold = i < n_enc or df.index[i - n_enc] in negrita_filas
                r.font.color.rgb = TINTA
    if dos:  # combina el encabezado superior repetido
        j = 1
        while j < n_c:
            k = j
            while k + 1 < n_c and df.columns[k][0] == df.columns[j - 1][0]:
                k += 1
            if k > j:
                for jj in range(j + 1, k + 1):   # vaciar antes de combinar: merge junta los textos
                    t.cell(0, jj).text = ""
                t.cell(0, j).merge(t.cell(0, k))
            j = k + 1
    return forma


def a_texto(t: pd.DataFrame, filas_indicador=False) -> pd.DataFrame:
    """Formatea una tabla: por fila (indicadores en filas) o por columna (nivel 0)."""
    out = t.astype(object).copy()
    for c in t.columns:
        for i in t.index:
            ind = i if filas_indicador else (c[0] if isinstance(c, tuple) else c)
            v = t.at[i, c]
            if isinstance(v, str):          # columnas de texto (p. ej. canal de venta)
                continue
            out.at[i, c] = variacion(v, ind.startswith("%")) if es_variacion(c) else formato_de(ind)(v)
    return out


def corto(ind: str) -> str:
    """Rotulo corto para encabezados de tabla."""
    return ind.replace("% aplicado al precio de lista", "% aplicado").replace("de precio de mercado", "precio mercado")

## C.4 Informe de un cliente

Cuatro láminas para `CLIENTE` (A.1), desde sus líneas de factura (enero 2025 al último mes con datos):

1. **Resumen**: la tabla comparativa del acumulado 2025 vs 2026 (igual a la de las sociedades).
2. **Tendencia del % de ajuste** mes a mes (fecha corta + ajuste de precio de mercado = general) y la venta bruta vs neta.
3. **Materiales con más ajuste**: los `TOP_MATERIALES` de mayor ajuste general en el último año, 2025 vs 2026.
4. **Mes a mes**: la tabla mensual con mapa de calor en los % de ajuste.

In [ ]:
SQL_CLIENTE = """
SELECT v.date AS fecha, v.bill AS factura, v.material, m.name AS producto, g6.name AS grupo_material,
       cdd.company_code AS sociedad, regexp_replace(cdd.new_zone_name, '^[^-]*-', '') AS distribuidora,
       dc.distribution_channel_name AS canal, v.kilogram AS kg, v.value AS neta,
       v.value + COALESCE(v.discount_value, 0) AS bruta,
       COALESCE(v.discount_market_price_adjustment, 0) AS aj,
       COALESCE(v.short_dated_discount, 0) AS fc
FROM public.daily_sales_sector_10_15_20_21 v
LEFT JOIN public.materials m ON v.material::integer = m.code
LEFT JOIN public.material_groups_6 g6 ON g6.code = m.gr6
LEFT JOIN public.commercial_distributors_details cdd ON v.id_commercial_distributors = cdd.id
LEFT JOIN public.distribution_channels dc ON v.distribution_channel = dc.distribution_channel
WHERE v.client = :cliente AND v.date >= :ini AND v.date < :fin
"""
with ENGINE.connect() as c:
    cl = pd.read_sql(text(SQL_CLIENTE), c, params={"cliente": CLIENTE, "ini": f"{A1}-01-01",
                                                   "fin": (FECHA_MAX + pd.Timedelta(days=1)).strftime("%Y-%m-%d")})
for col in ["kg", "neta", "bruta", "aj", "fc"]:
    cl[col] = cl[col].astype(float)
cl["fecha"] = pd.to_datetime(cl["fecha"])
cl["gen"] = cl["aj"] + cl["fc"]
cl["Mes"] = cl["fecha"].dt.to_period("M")
cl["Año"] = cl["fecha"].dt.year
NOMBRE_CLI = nombre_cliente.get(CLIENTE, CLIENTE).strip()
SOC_CLI = SOCIEDADES.get(str(cl["sociedad"].mode().iloc[0]), "")


def ind_cliente(df, por) -> pd.DataFrame:
    """Indicadores del cliente desde sumas."""
    g = df.groupby(por)[["kg", "bruta", "neta", "aj", "fc", "gen"]].sum()
    b = g["bruta"].where(g["bruta"] != 0)
    kg = g["kg"].where(g["kg"] > 0)
    return pd.DataFrame({
        "Facturas": df.groupby(por)["factura"].nunique(),
        VENTA_KG: g["kg"],
        VENTA_BRUTA: g["bruta"], VENTA_NETA: g["neta"], P_GEN: g["gen"],
        P_AJ: g["aj"], P_FC: g["fc"],
        D_GEN: g["gen"] / b * 100, D_AJ: g["aj"] / b * 100, D_FC: g["fc"] / b * 100,
        PRECIO_LISTA: g["bruta"] / kg, PRECIO_NETO: g["neta"] / kg,
    })


acum_cli = cl[cl["fecha"].dt.month.isin(MESES_ACUM)]
anual_cli = ind_cliente(acum_cli, "Año").reindex(ANIOS)
mes_cli_t = ind_cliente(cl, "Mes")
kg_fc = acum_cli[acum_cli["fc"] > 0].groupby("Año")["kg"].sum() / acum_cli.groupby("Año")["kg"].sum() * 100
# materiales: top por ajuste general del ultimo año
mat_cli = ind_cliente(acum_cli, ["producto", "Año"])
top_mat = (mat_cli.xs(A2, level="Año")[P_GEN].sort_values(ascending=False).head(TOP_MATERIALES).index
           if A2 in mat_cli.index.get_level_values("Año") else [])
print(f"{CLIENTE} - {NOMBRE_CLI} · {SOC_CLI} · {len(cl):,} líneas · {cl['factura'].nunique():,} facturas")

In [ ]:
def figura_tendencia_cliente(figsize=(14, 7.6)):
    """Barras apiladas del % de ajuste por mes (fecha corta + ajuste = general) y venta bruta vs neta (la franja es
    el ajuste)."""
    t = mes_cli_t
    x = list(range(len(t)))
    etiquetas = t.index.strftime("%Y-%m")
    corte = sum(t.index.year == A1) - 0.5
    fig, (a1, a2) = plt.subplots(2, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [1.7, 1]})
    a1.bar(x, t[D_FC], color=MOTIVOS["Con fecha corta"], width=0.75, edgecolor="white", linewidth=1, label="Fecha corta")
    a1.bar(x, t[D_AJ], bottom=t[D_FC], color=MOTIVOS["Solo ajuste de precio de mercado"], width=0.75, edgecolor="white", linewidth=1,
           label="Ajuste de precio de mercado")
    for i, v in enumerate(t[D_GEN]):
        a1.annotate(pct(v), (i, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8,
                    color="#0b0b0b", fontweight="bold")
    a1.set_ylim(0, max(100, t[D_GEN].max() * 1.12))
    a1.set_title("% de ajuste sobre la venta bruta, por tipo (la suma es el % de ajuste general)", loc="left", fontsize=11)
    a1.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1, 1))
    color_ = COLOR_SOC.get(SOC_CLI, "#2a78d6")
    a2.fill_between(x, t[VENTA_NETA], t[VENTA_BRUTA], color=TINTE_SOC.get(SOC_CLI, "#a9c9f0"), alpha=0.6,
                    linewidth=0, label="Ajuste")
    a2.plot(x, t[VENTA_BRUTA], color=color_, linewidth=2, marker="o", markersize=4, label="Venta bruta")
    a2.plot(x, t[VENTA_NETA], color=color_, linewidth=2, linestyle="--", marker="o", markersize=4, label="Venta neta")
    for i in (0, len(x) - 1):
        a2.annotate(pesos(t[VENTA_BRUTA].iloc[i]), (x[i], t[VENTA_BRUTA].iloc[i]), xytext=(0, 6),
                    textcoords="offset points", ha="center", fontsize=8, color="#0b0b0b")
        a2.annotate(pesos(t[VENTA_NETA].iloc[i]), (x[i], t[VENTA_NETA].iloc[i]), xytext=(0, -10),
                    textcoords="offset points", ha="center", va="top", fontsize=8, color="#0b0b0b")
    a2.margins(y=0.25)
    a2.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1, 1))
    a2.set_title("Venta bruta y venta neta por mes ($) — la franja es el ajuste", loc="left", fontsize=11)
    for ax, f in ((a1, lambda v, _: f"{v:,.0f}%"), (a2, lambda v, _: pesos(v))):
        ax.axvline(corte, color="#8c8b87", linestyle=":", linewidth=1)
        ejes_limpios(ax, formato_y=f)
    a2.set_xticks(x, etiquetas, rotation=90)
    plt.tight_layout()
    return fig


def figura_materiales_cliente(figsize=(14, 7)):
    """Top materiales por ajuste general: % de ajuste por tipo (barras apiladas), un panel por año; al lado % y $."""
    fig, ejes = plt.subplots(1, len(ANIOS), figsize=figsize, sharey=True, sharex=True)
    nombres = [str(m).title()[:34] for m in top_mat]
    for ax, anio in zip(ejes, ANIOS):
        t = mat_cli[mat_cli.index.get_level_values("Año") == anio].droplevel("Año").reindex(top_mat)
        fc, aj = t[D_FC].fillna(0), t[D_AJ].fillna(0)
        ax.barh(nombres, fc, color=MOTIVOS["Con fecha corta"], height=0.7, edgecolor="white", label="Fecha corta")
        ax.barh(nombres, aj, left=fc, color=MOTIVOS["Solo ajuste de precio de mercado"], height=0.7, edgecolor="white", label="Ajuste de precio de mercado")
        for i, (g_, plata, kg_) in enumerate(zip(t[D_GEN], t[P_GEN], t[VENTA_KG])):
            texto_ = "sin compras" if pd.isna(g_) else f"{pct(g_)} · {pesos(plata)} · {num(kg_, 0)} kg"
            ax.annotate(texto_, (0 if pd.isna(g_) else g_, i), xytext=(4, 0), textcoords="offset points",
                        va="center", fontsize=7.5, color="#52514e")
        ax.set_xlim(0, 175)
        ax.set_title(str(anio), loc="left", fontsize=12)
        ejes_limpios(ax, formato_x=lambda v, _: f"{v:,.0f}%" if v <= 100 else "")
    ejes[0].invert_yaxis()
    ejes[-1].legend(frameon=False, fontsize=9, loc="lower right")
    fig.suptitle(f"% de ajuste por tipo · al lado: % ajuste general · ajuste en pesos · venta kg ({PERIODO})",
                 x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    return fig


def tabla_mensual_cliente() -> pd.DataFrame:
    """Mes a mes + totales del acumulado de cada año."""
    t = mes_cli_t.copy()
    t.index = t.index.strftime("%Y-%m")
    tot = anual_cli.copy()
    tot.index = [f"{PERIODO.capitalize()} {a_}" for a_ in tot.index]
    return pd.concat([t, tot])[["Facturas", VENTA_KG, VENTA_BRUTA, VENTA_NETA, P_GEN,
                                D_GEN, D_AJ, D_FC, PRECIO_LISTA, PRECIO_NETO]]


def construir_informe_cliente():
    nueva_presentacion()
    a1, a2 = anual_cli.loc[A1], anual_cli.loc[A2]
    seccion = f"{CLIENTE} · {NOMBRE_CLI}"

    # ------------------------------------------------------------------ 1. resumen
    d = diapositiva(f"{NOMBRE_CLI.title()}: el {pct(kg_fc.get(A2, 0))} de lo que compra es fecha corta y paga "
                    f"{pct(100 - a2[D_GEN])} de la lista",
                    f"{SOC_CLI} · {', '.join(cl['distribuidora'].dropna().unique()[:2])} · "
                    f"{', '.join(cl['canal'].dropna().unique()[:2])} · {PERIODO} {A2} vs {A1}", seccion)
    # misma tabla comparativa del panorama de las sociedades
    sumas = (acum_cli.rename(columns={"kg": "Venta Kg", "bruta": "Venta Bruta", "neta": "Venta Neta",
                                      "aj": "Descuento Ajuste Precio Mercado", "fc": "Descuento Rotacion / Fecha Corta"})
             .groupby("Año")[MEDIDAS].sum().reindex(ANIOS))
    pan = a_texto(comparar(indicadores(sumas)), filas_indicador=True)
    pan.index.name = "Indicador"
    tabla(d, pan, x=Inches(1.5), w=Inches(10.3), tam=11, ancho_indice=0.42,
          negrita_filas=[PRECIO_LISTA, PRECIO_NETO, L_GEN, L_AJ, L_FC], alto_fila=Inches(0.33))

    # ------------------------------------------------------------------ 2. tendencia
    m2 = mes_cli_t[mes_cli_t.index.year == A2]
    d = diapositiva(f"El % de ajuste pasó de {pct(a1[D_GEN])} a {pct(a2[D_GEN])}; el máximo fue "
                    f"{pct(m2[D_GEN].max())} en {MESES_ES[m2[D_GEN].idxmax().month - 1]} de {A2}",
                    "Barras: % de ajuste por mes (amarillo = fecha corta, verde = ajuste); abajo, venta bruta y neta",
                    seccion)
    imagen(d, figura_tendencia_cliente(figsize=(15, 7.4)))

    # ------------------------------------------------------------------ 3. materiales
    top1 = top_mat[0]
    share_top1 = mat_cli.loc[(top1, A2), P_GEN] / a2[P_GEN] * 100
    d = diapositiva(f"{str(top1).title()} concentra el {pct(share_top1)} del ajuste, con "
                    f"{pct(mat_cli.loc[(top1, A2), D_FC])} de fecha corta",
                    f"Los {len(top_mat)} materiales con más ajuste general en {PERIODO} {A2}, comparados con {A1}", seccion)
    imagen(d, figura_materiales_cliente(figsize=(15, 7.2)))

    # ------------------------------------------------------------------ 4. mensual
    tm = tabla_mensual_cliente()
    txt = a_texto(tm)
    txt["Facturas"] = tm["Facturas"].map(lambda v: num(v, 0))
    txt.columns = [c.replace(D_AJ, "% Ajuste precio mercado").replace(PRECIO_LISTA, "Lista/kg").replace(PRECIO_NETO, "Neto/kg")
                   for c in tm.columns]
    txt.index.name = "Mes"
    calor = tm[[D_GEN, D_AJ, D_FC]].reindex(columns=tm.columns)
    calor.iloc[-2:] = float("nan")   # los totales no se pintan
    calor.index = txt.index
    d = diapositiva(f"Mes a mes: desde enero de {A2} el ajuste no baja de {pct(m2[D_GEN].min())} y el neto por kilo "
                    f"terminó en {pesos(m2[PRECIO_NETO].iloc[-1])}",
                    "Valores enteros en pesos · mapa de calor en los % de ajuste · al final, los totales del acumulado",
                    seccion)
    tabla(d, txt, y=Inches(1.5), tam=8, ancho_indice=0.11, alto_fila=Inches(0.215),
          negrita_filas=list(txt.index[-2:]), calor=calor)

    guardar(f"Informe_Cliente_{CLIENTE}_{A1}_vs_{A2}.pptx")


construir_informe_cliente()